# Tech Challenge Fiap - Fase 1

## Case NPS Preditivo
Prever satisfação do cliente (NPS) antes da aplicação da pesquisa

# Fase 5 e 6 - Evaluation e Deployment

### **Conclusão Executiva**

O projeto respondeu ao desafio central: é possível prever se um cliente será Detrator antes da pesquisa NPS, utilizando apenas dados operacionais disponíveis no momento da entrega.

Todas as metas técnicas definidas na Fase 1 foram atingidas pelo modelo escolhido: Random Forest Classificador para prever se um cliente será Detrator ou não e Regressão Linear para prever o NPS score de cliente. 

**O que os dados revelaram?** 

A análise exploratória identificou que a insatisfação do cliente está concentrada em dois fatores operacionais principais: número de reclamações (complaints_count) e dias de atraso na entrega (delivery_delay_days), que juntos representam quase 40% do poder preditivo do modelo.

Variáveis como valor do pedido e região geográfica não apresentaram associação relevante com o NPS, o problema é da qualidade de serviço, não de perfil do cliente.

**Modelo escolhido e resultado** 

| Modelo | Métrica | Meta | Resultado |
|---|---|---|---|
| Random Forest | Recall | ≥ 75% | 95% ✅ |
| Random Forest | AUC-ROC | ≥ 0.80 | 85% ✅ |
| Random Forest | F1-Score | ≥ 0.70 | 88% ✅ |
| Random Forest | Precision | ≥ 60% | 82% ✅ |

**Impacto esperado para o negócio**

Com Recall de 95%, o modelo identifica preventivamente 95 de cada 100 futuros Detratores antes da pesquisa NPS. 

Isso permite que a equipe de CRM atue proativamente: com cupons, contato personalizado ou resolução de problemas antes que o cliente avalie negativamente a empresa.

A meta de negócio definida na Fase 1: reduzir Detratores em 10% em 90 dias é factível com esse modelo em produção, considerando que ações preventivas têm efetividade comprovada em programas de experiência do cliente.

**Próximos passos recomendados**

1. Deploy do modelo: integrar o Random Forest ao pipeline operacional para gerar scores em até 24h após a entrega.

2. Teste A/B: comparar grupo com ação preventiva vs grupo controle para medir efetividade real.

3. Monitoramento: acompanhar o Recall e AUC-ROC mensalmente para detectar target drift.

4. Retreinamento: atualizar o modelo trimestralmente com novos dados para manter a performance.


### **Deployment: Demonstração em Produção**

O modelo treinado na Fase 4 é salvo em disco usando joblib — 
uma biblioteca de serialização do Python. Em produção, esse arquivo 
seria carregado diretamente em uma API ou sistema interno sem 
necessidade de retreinar.


In [7]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Carregando os dados de classificação
X_clf = pd.read_csv('X_train.csv')
y_clf = pd.read_csv('y_train.csv').squeeze()  # nps_detrator (0 ou 1)

# Treinando o Random Forest Classificador — mesmos parâmetros da Fase 4
modelo_rf_clf = RandomForestClassifier(
    class_weight='balanced',  # compensa desbalanceamento 74/26
    random_state=42           # garante reprodutibilidade
)
modelo_rf_clf.fit(X_clf, y_clf)

# Salvando o modelo correto em disco
joblib.dump(modelo_rf_clf, 'modelo_rf_classificador.pkl')
print(f"✅ Modelo salvo: {type(modelo_rf_clf)}")

# Nota: o modelo é retreinado aqui com os mesmos parâmetros da Fase 4
# pois o objeto original foi sobrescrito por modelos subsequentes no Notebook 04.
# Em produção, o modelo seria salvo imediatamente após o treino.

✅ Modelo salvo: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


### **Simulação de uso em Produção**

Demonstração do fluxo completo: um novo cliente conclui  a jornada de compra e o sistema retorna imediatamente se ele será Detrator e com qual probabilidade.

In [8]:
# Carregando o modelo salvo — simula uso em produção
modelo_producao = joblib.load('modelo_rf_classificador.pkl')

# Simulando novo cliente chegando ao sistema
novo_cliente = pd.read_csv('X_test.csv').iloc[[0]]

# Previsão
previsao = modelo_producao.predict(novo_cliente)
probabilidade = modelo_producao.predict_proba(novo_cliente)

print(f"Previsão: {'Detrator' if previsao[0] == 1 else 'Não Detrator'}")
print(f"Probabilidade de ser Detrator: {probabilidade[0][1]:.2%}")

Previsão: Detrator
Probabilidade de ser Detrator: 89.00%


O modelo classificou esse cliente como Detrator com 89% de probabilidade: alto grau de certeza. Em produção, esse cliente seria imediatamente acionado pela equipe de CRM.

Esse é o fluxo completo em produção:

1. Cliente conclui a entrega
2. Sistema coleta os dados operacionais automaticamente  
3. Modelo retorna a classificação e probabilidade em tempo real
4. Equipe de CRM é acionada para clientes com probabilidade acima do threshold definido